<a href="https://colab.research.google.com/github/romero-sebastian/econ3916-statsml/blob/main/Lab13/Lab_13_Hedonic_Pricing_%26_FWL_Theorem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/_____/Data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

# 3c: Regress Residuals on Residuals (-1 removes intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals - 1', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])

import numpy as np
import plotly.graph_objects as go

# --- Extract coefficients from the multivariate model ---
intercept = multi_model.params['Intercept']
b_age      = multi_model.params['Property_Age']
b_dist     = multi_model.params['Distance_to_Tech_Hub']

# --- Build a meshgrid spanning the range of both predictors ---
# This creates a 2D grid of (Age, Distance) combinations so we can
# evaluate the plane equation at every point on the surface.
age_range  = np.linspace(df['Property_Age'].min(),        df['Property_Age'].max(),        50)
dist_range = np.linspace(df['Distance_to_Tech_Hub'].min(), df['Distance_to_Tech_Hub'].max(), 50)

age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# --- Plane equation: ŷ = intercept + b1*Age + b2*Distance ---
# Each cell in price_grid is the model's predicted price for that
# (Age, Distance) pair — this is the actual regression hyperplane.
price_grid = intercept + b_age * age_grid + b_dist * dist_grid

# --- Build the figure ---
fig = go.Figure()

# Scatter: actual data points
fig.add_trace(go.Scatter3d(
    x=df['Property_Age'],
    y=df['Distance_to_Tech_Hub'],
    z=df['Sale_Price'],
    mode='markers',
    marker=dict(size=3, color=df['Sale_Price'], colorscale='Viridis', opacity=0.6),
    name='Actual Sales'
))

# Surface: the fitted regression hyperplane
fig.add_trace(go.Surface(
    x=age_grid,
    y=dist_grid,
    z=price_grid,
    opacity=0.55,
    colorscale='RdBu',
    name='Regression Hyperplane',
    showscale=False
))

fig.update_layout(
    title='Hedonic Pricing: OLS Hyperplane in 3D Feature Space',
    scene=dict(
        xaxis_title='Property Age (years)',
        yaxis_title='Distance to Tech Hub (miles)',
        zaxis_title='Sale Price ($)'
    ),
    legend=dict(x=0.01, y=0.99)
)

fig.show()

HTTPError: HTTP Error 400: Bad Request